In [ ]:
import random
import time

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from scipy.stats import pearsonr, spearmanr
from transformers import AutoModel, AutoTokenizer

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

model_name = "distilbert-base-uncased"
dataset_name = "glue"
dataset_config = "stsb"
split_name = "validation"
device = "mps" if torch.backends.mps.is_available() else "cpu"
max_length = 128
batch_size = 256 if device == "mps" else 64
start_time = time.time()

print({
    "model_name": model_name,
    "dataset": f"{dataset_name}/{dataset_config}",
    "split": split_name,
    "device": device,
    "batch_size": batch_size,
    "max_length": max_length,
    "seed": seed,
})

In [ ]:
ds = load_dataset(dataset_name, dataset_config, split=split_name)
df = ds.to_pandas()[["sentence1", "sentence2", "label"]].copy()
print({"num_examples": len(df), "columns": df.columns.tolist()})
print(df.head())

In [ ]:
sentences1 = df["sentence1"].astype(str).tolist()
sentences2 = df["sentence2"].astype(str).tolist()
labels = df["label"].to_numpy(dtype=np.float32)

all_sentences = sentences1 + sentences2
unique_sentences = list(dict.fromkeys(all_sentences))
sentence_to_idx = {text: idx for idx, text in enumerate(unique_sentences)}

pair_idx1 = np.fromiter((sentence_to_idx[s] for s in sentences1), dtype=np.int32, count=len(sentences1))
pair_idx2 = np.fromiter((sentence_to_idx[s] for s in sentences2), dtype=np.int32, count=len(sentences2))

total_sentence_occurrences = len(all_sentences)
num_unique_sentences = len(unique_sentences)
cache_hits = total_sentence_occurrences - num_unique_sentences
cache_hit_rate = cache_hits / total_sentence_occurrences if total_sentence_occurrences else 0.0

print({
    "total_sentence_occurrences": total_sentence_occurrences,
    "num_unique_sentences": num_unique_sentences,
    "cache_hits": cache_hits,
    "cache_hit_rate": round(cache_hit_rate, 6),
})

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.to(device)
model.eval()

print({
    "model_name": model_name,
    "hidden_size": int(model.config.hidden_size),
    "device": device,
})

In [ ]:
unique_embeddings_list = []
unique_token_lengths = []

with torch.no_grad():
    for start in range(0, len(unique_sentences), batch_size):
        batch_sentences = unique_sentences[start:start + batch_size]
        encoded = tokenizer(
            batch_sentences,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        attention_mask = encoded["attention_mask"]
        token_lengths = attention_mask.sum(dim=1).cpu().numpy().astype(np.int32)
        unique_token_lengths.append(token_lengths)

        encoded = {k: v.to(device) for k, v in encoded.items()}
        outputs = model(**encoded)
        cls_embeddings = outputs.last_hidden_state[:, 0, :]
        cls_embeddings = torch.nn.functional.normalize(cls_embeddings, p=2, dim=1)
        unique_embeddings_list.append(cls_embeddings.cpu().numpy().astype(np.float32))

unique_embeddings = np.vstack(unique_embeddings_list)
unique_token_lengths = np.concatenate(unique_token_lengths)
unique_embedding_norms = np.linalg.norm(unique_embeddings, axis=1)

print({
    "embedding_shape": tuple(unique_embeddings.shape),
    "num_token_lengths": int(len(unique_token_lengths)),
    "norm_min": float(unique_embedding_norms.min()),
    "norm_max": float(unique_embedding_norms.max()),
})

In [ ]:
emb1 = unique_embeddings[pair_idx1]
emb2 = unique_embeddings[pair_idx2]

cosine_similarity = np.sum(emb1 * emb2, axis=1)
predicted_score_0_5 = 2.5 * (cosine_similarity + 1.0)

results_df = df.copy()
results_df["sentence1_idx"] = pair_idx1
results_df["sentence2_idx"] = pair_idx2
results_df["sentence1_token_length"] = unique_token_lengths[pair_idx1]
results_df["sentence2_token_length"] = unique_token_lengths[pair_idx2]
results_df["sentence1_embedding_norm"] = unique_embedding_norms[pair_idx1]
results_df["sentence2_embedding_norm"] = unique_embedding_norms[pair_idx2]
results_df["cosine_similarity"] = cosine_similarity
results_df["predicted_score_0_5"] = predicted_score_0_5

print(results_df[[
    "sentence1",
    "sentence2",
    "label",
    "sentence1_token_length",
    "sentence2_token_length",
    "cosine_similarity",
    "predicted_score_0_5",
]].head(10))

In [ ]:
pearson_corr = pearsonr(predicted_score_0_5, labels).statistic
spearman_corr = spearmanr(predicted_score_0_5, labels).statistic

embedding_norm_summary = pd.DataFrame({
    "group": ["unique_sentences", "sentence1_in_pairs", "sentence2_in_pairs", "all_pair_positions"],
    "count": [
        len(unique_embedding_norms),
        len(results_df),
        len(results_df),
        len(results_df) * 2,
    ],
    "mean": [
        float(np.mean(unique_embedding_norms)),
        float(np.mean(results_df["sentence1_embedding_norm"])),
        float(np.mean(results_df["sentence2_embedding_norm"])),
        float(np.mean(np.concatenate([
            results_df["sentence1_embedding_norm"].to_numpy(),
            results_df["sentence2_embedding_norm"].to_numpy(),
        ]))),
    ],
    "std": [
        float(np.std(unique_embedding_norms)),
        float(np.std(results_df["sentence1_embedding_norm"])),
        float(np.std(results_df["sentence2_embedding_norm"])),
        float(np.std(np.concatenate([
            results_df["sentence1_embedding_norm"].to_numpy(),
            results_df["sentence2_embedding_norm"].to_numpy(),
        ]))),
    ],
    "min": [
        float(np.min(unique_embedding_norms)),
        float(np.min(results_df["sentence1_embedding_norm"])),
        float(np.min(results_df["sentence2_embedding_norm"])),
        float(np.min(np.concatenate([
            results_df["sentence1_embedding_norm"].to_numpy(),
            results_df["sentence2_embedding_norm"].to_numpy(),
        ]))),
    ],
    "median": [
        float(np.median(unique_embedding_norms)),
        float(np.median(results_df["sentence1_embedding_norm"])),
        float(np.median(results_df["sentence2_embedding_norm"])),
        float(np.median(np.concatenate([
            results_df["sentence1_embedding_norm"].to_numpy(),
            results_df["sentence2_embedding_norm"].to_numpy(),
        ]))),
    ],
    "max": [
        float(np.max(unique_embedding_norms)),
        float(np.max(results_df["sentence1_embedding_norm"])),
        float(np.max(results_df["sentence2_embedding_norm"])),
        float(np.max(np.concatenate([
            results_df["sentence1_embedding_norm"].to_numpy(),
            results_df["sentence2_embedding_norm"].to_numpy(),
        ]))),
    ],
})

all_pair_token_lengths = np.concatenate([
    results_df["sentence1_token_length"].to_numpy(),
    results_df["sentence2_token_length"].to_numpy(),
]).astype(np.int32)

token_length_summary = pd.DataFrame({
    "group": ["unique_sentences", "sentence1_in_pairs", "sentence2_in_pairs", "all_pair_positions"],
    "count": [
        len(unique_token_lengths),
        len(results_df),
        len(results_df),
        len(all_pair_token_lengths),
    ],
    "mean": [
        float(np.mean(unique_token_lengths)),
        float(np.mean(results_df["sentence1_token_length"])),
        float(np.mean(results_df["sentence2_token_length"])),
        float(np.mean(all_pair_token_lengths)),
    ],
    "std": [
        float(np.std(unique_token_lengths)),
        float(np.std(results_df["sentence1_token_length"])),
        float(np.std(results_df["sentence2_token_length"])),
        float(np.std(all_pair_token_lengths)),
    ],
    "min": [
        int(np.min(unique_token_lengths)),
        int(np.min(results_df["sentence1_token_length"])),
        int(np.min(results_df["sentence2_token_length"])),
        int(np.min(all_pair_token_lengths)),
    ],
    "median": [
        float(np.median(unique_token_lengths)),
        float(np.median(results_df["sentence1_token_length"])),
        float(np.median(results_df["sentence2_token_length"])),
        float(np.median(all_pair_token_lengths)),
    ],
    "max": [
        int(np.max(unique_token_lengths)),
        int(np.max(results_df["sentence1_token_length"])),
        int(np.max(results_df["sentence2_token_length"])),
        int(np.max(all_pair_token_lengths)),
    ],
})

print("embedding_norm_summary")
print(embedding_norm_summary)
print("token_length_summary")
print(token_length_summary)

In [ ]:
runtime_seconds = time.time() - start_time

print(f"device_used: {device}")
print(f"model_name: {model_name}")
print(f"dataset_split: {dataset_name}/{dataset_config}/{split_name}")
print(f"num_examples: {len(df)}")
print(f"total_sentence_occurrences: {total_sentence_occurrences}")
print(f"num_unique_sentences: {num_unique_sentences}")
print(f"cache_hits: {cache_hits}")
print(f"cache_hit_rate: {cache_hit_rate:.6f}")
print(f"pearson_correlation: {pearson_corr:.6f}")
print(f"spearman_correlation: {spearman_corr:.6f}")
print("embedding_norm_summary_records:")
print(embedding_norm_summary.to_dict(orient="records"))
print("token_length_summary_records:")
print(token_length_summary.to_dict(orient="records"))
print(f"runtime_seconds: {runtime_seconds:.2f}")